In [1]:
import os
import torch
import onnx
import json
from onnxsim import simplify
from onnxconverter_common import float16
from transformers import AutoModelForCausalLM, AutoProcessor

os.environ["ONNXSIM_FIXED_POINT_ITERS"] = "200"

class Florence2ONNXExporter:
    # ─────────────────────────────────────────────
    # LOAD MODEL
    # ─────────────────────────────────────────────
    def __init__(self, model_path, device="cpu"):
        
        self.device = device

        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            trust_remote_code=True
        ).eval().to(device)

        self.processor = AutoProcessor.from_pretrained(
            model_path,
            trust_remote_code=True
        )
        
        self.img_size = self.processor.image_processor.crop_size['width']

    # ─────────────────────────────────────────────
    # WRAPPERS
    # ─────────────────────────────────────────────
    class EncoderWrapper(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model

        def forward(self, input_ids, pixel_values):
            image_features = self.model._encode_image(pixel_values)

            encoder = self.model.language_model.model.encoder
            text_embeds = encoder.embed_tokens(input_ids)

            inputs_embeds = torch.cat([image_features, text_embeds], dim=1)
            enc_out = encoder(inputs_embeds=inputs_embeds)

            return enc_out.last_hidden_state

    class DecoderWrapper(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.decoder = model.language_model.model.decoder
            self.lm_head = model.language_model.lm_head

        def forward(self, decoder_input_ids, encoder_hidden_states):
            dec_out = self.decoder(
                input_ids=decoder_input_ids,
                encoder_hidden_states=encoder_hidden_states,
                use_cache=False
            )
            return self.lm_head(dec_out.last_hidden_state)

    # ─────────────────────────────────────────────
    # DUMMY INPUTS
    # ─────────────────────────────────────────────
    def _prepare_dummy(self, batch, seq, img_size):
        self.dummy_input_ids = torch.randint(
            0, 1000, (batch, seq), dtype=torch.long
        ).to(self.device)

        self.dummy_pixel_values = torch.randn(
            batch, 3, img_size, img_size
        ).to(self.device)

        enc_wrapper = self.EncoderWrapper(self.model).eval().to(self.device)

        with torch.no_grad():
            enc_out = enc_wrapper(
                self.dummy_input_ids,
                self.dummy_pixel_values
            )

        enc_len = enc_out.shape[1]
        hidden_dim = enc_out.shape[2]

        self.dummy_enc_hidden = torch.randn(
            batch, enc_len, hidden_dim
        ).to(self.device)

        self.dummy_dec_ids = torch.ones(
            batch, 1, dtype=torch.long
        ).to(self.device)

        return enc_len, hidden_dim

    # ─────────────────────────────────────────────
    # UTILITIES
    # ─────────────────────────────────────────────
    @staticmethod
    def _convert_fp16(path):
        model = onnx.load(path)
        model_fp16 = float16.convert_float_to_float16(
            model,
            keep_io_types=False
        )
        onnx.save(model_fp16, path)

    def _attach_tokenizer(self, onnx_path):
        model = onnx.load(onnx_path)

        tokenizer_blob = json.loads(
            self.processor.tokenizer.backend_tokenizer.to_str()
        )

        tokenizer_blob["meta"] = {
            "special_ids": self.processor.tokenizer.all_special_ids,
            "bos_token_id": self.processor.tokenizer.bos_token_id,
            "eos_token_id": self.processor.tokenizer.eos_token_id,
            "pad_token_id": self.processor.tokenizer.pad_token_id,
        }

        meta = model.metadata_props.add()
        meta.key = "tokenizer_json"
        meta.value = json.dumps(tokenizer_blob)

        onnx.save(model, onnx_path)

    def _export(
        self,
        wrapper,
        inputs,
        input_names,
        output_names,
        dynamic_axes,
        path,
        simplify_model
    ):
        with torch.no_grad():
            torch.onnx.export(
                wrapper,
                inputs,
                path,
                input_names=input_names,
                output_names=output_names,
                dynamic_axes=dynamic_axes,
                opset_version=18,
                do_constant_folding=True,
                dynamo=False
            )

        print(f"Exported: {path}")

        if simplify_model:
            print("Simplifying...")
            model_onnx = onnx.load(path)
            model_simp, check = simplify(model_onnx)

            if check:
                onnx.save(model_simp, path)
                self._convert_fp16(path)
                print("Simplified ✔")
            else:
                print("Simplification failed")

    # ─────────────────────────────────────────────
    # RUN (ALL PARAMS HERE)
    # ─────────────────────────────────────────────
    def run(
        self,
        encoder_out="encoder.onnx",
        decoder_out="decoder.onnx",
        batch=1,
        seq=32,
        simplify_model=True,
        dynamic_batch=True
    ):
        # wrappers
        enc_wrapper = self.EncoderWrapper(self.model).eval().to(self.device)
        dec_wrapper = self.DecoderWrapper(self.model).eval().to(self.device)

        # dummy
        enc_len, hidden_dim = self._prepare_dummy(batch, seq, self.img_size)

        encoder_dynamic_axes = {
            "input_ids": {0: "batch", 1: "seq"},
            "pixel_values": {0: "batch"},
            "encoder_hidden_states": {0: "batch", 1: "enc_len"},
        }

        if not dynamic_batch:
            encoder_dynamic_axes["input_ids"] = {1: "seq"}
            encoder_dynamic_axes["pixel_values"] = {}
            encoder_dynamic_axes["encoder_hidden_states"] = {1: "enc_len"}
        
        self._export(
            enc_wrapper,
            (self.dummy_input_ids, self.dummy_pixel_values),
            ["input_ids", "pixel_values"],
            ["encoder_hidden_states"],
            encoder_dynamic_axes,
            encoder_out,
            simplify_model
        )

        decoder_dynamic_axes = {
            "decoder_input_ids": {0: "batch", 1: "dec_len"},
            "encoder_hidden_states": {0: "batch", 1: "enc_len"},
            "logits": {0: "batch", 1: "dec_len"},
        }

        if not dynamic_batch:
            decoder_dynamic_axes["decoder_input_ids"] = {1: "dec_len"}
            decoder_dynamic_axes["encoder_hidden_states"] = {1: "enc_len"}
            decoder_dynamic_axes["logits"] = {1: "dec_len"}

        self._export(
            dec_wrapper,
            (self.dummy_dec_ids, self.dummy_enc_hidden),
            ["decoder_input_ids", "encoder_hidden_states"],
            ["logits"],
            decoder_dynamic_axes,
            decoder_out,
            simplify_model
        )

        # attach tokenizer to decoder
        self._attach_tokenizer(decoder_out)

        print("\nDone.")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
exporter = Florence2ONNXExporter(model_path="./weights/model_best_st_224x224")
exporter.run(
    encoder_out="florence2_encoder_fp16.onnx",
    decoder_out="florence2_decoder_fp16.onnx",
)